# Black–Litterman as Bayesian Updating

08/01 showed that historical means make mean-variance optimization useless, and 08/03 avoided means altogether. Black and
Litterman (1992) kept the means but changed where they come from. Start from the returns that make the market portfolio
optimal (a prior), add a few views with stated confidence (data), and combine them with Bayes' theorem. The result is a set of
expected returns that stays close to equilibrium where you have no opinion and tilts where you do, and portfolios that look
like the market plus a few deliberate bets.

## 0. Math link

- [From MLE to MAP](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/05_point_estimation/06_from_mle_to_map.ipynb)
  and [Bayesian updating](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/05_point_estimation/09_bayesian_updating.ipynb):
  a normal prior combined with normal data gives a normal posterior whose mean is a precision-weighted average.
- [Bayes' theorem](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/00_introduction_to_probability/11_bayes_theorem_formula.ipynb).
- Reverse optimization $\Pi = \delta\Sigma w_M$ from [04/05](../04_portfolio_mathematics/05_capm_and_security_market_line.ipynb);
  the failure of plug-in means from [08/01](./01_why_mean_variance_optimization_breaks.ipynb).

## 1. Introduction

A CIO believes emerging-market equities will do well and Swiss small caps will beat Swiss large caps. A mean-variance optimizer
needs expected returns for all eight asset classes, not two opinions. Black–Litterman fills in the rest from market equilibrium
and blends in the opinions according to how confident the CIO is.

After this notebook you can:

- derive equilibrium expected returns by reverse optimization;
- write the Black–Litterman prior, views and posterior, and derive the posterior as a Bayesian update;
- compute posterior expected returns and portfolio weights for absolute and relative views;
- explain the roles of $\tau$ and $\Omega$ and how confidence moves the portfolio from the market to the views;
- use Black–Litterman to combine historical data with an equilibrium prior, and compare it with the plug-in portfolio.

## 2. Theory

### The prior: equilibrium returns

If all investors hold the market portfolio $w_M$ with risk aversion $\delta$, then $w_M = (\delta\Sigma)^{-1}\Pi$ for the expected excess returns $\Pi$, so

$$
\Pi = \delta\,\Sigma\,w_M
$$

$\delta$ can be set from the market: $\delta = (\text{market expected excess return})/\sigma_M^2$. The prior says the true excess returns are uncertain around $\Pi$:

$$
\mu \sim N(\Pi,\ \tau\Sigma)
$$

with a small scalar $\tau$ (often 0.025–0.05, or $1/T$ for $T$ years of data used to estimate $\Pi$).

### The views

$k$ views are linear statements about $\mu$ with noise:

$$
P\mu = q + \varepsilon, \qquad \varepsilon \sim N(0, \Omega)
$$

Each row of $P$ is a portfolio: an absolute view on asset $i$ has a 1 in column $i$; a relative view "asset $i$ beats asset $j$ by 1.5%" has $+1$ and $-1$
and $q = 1.5\%$. $\Omega$ (usually diagonal) holds the uncertainty of each view.

### The posterior

**Scalar case first.** Prior $\mu \sim N(\pi, s_0^2)$, one view $q$ with variance $\omega$. The posterior is normal with

$$
E[\mu \mid q] = \frac{\pi/s_0^2 + q/\omega}{1/s_0^2 + 1/\omega}, \qquad \operatorname{Var}(\mu \mid q) = \frac{1}{1/s_0^2 + 1/\omega}
$$

a precision-weighted average. **Matrix case.** Multiplying the two normal densities and completing the square:

$$
\mu_{BL} = \big[(\tau\Sigma)^{-1} + P^\top\Omega^{-1}P\big]^{-1}\big[(\tau\Sigma)^{-1}\Pi + P^\top\Omega^{-1}q\big], \qquad
M = \big[(\tau\Sigma)^{-1} + P^\top\Omega^{-1}P\big]^{-1}
$$

An equivalent form, easier to read and cheaper to compute:

$$
\mu_{BL} = \Pi + \tau\Sigma P^\top\big(P\tau\Sigma P^\top + \Omega\big)^{-1}(q - P\Pi)
$$

The update is proportional to the **surprise** $q - P\Pi$, the difference between the view and what equilibrium already implies. Through $\Sigma P^\top$ a view on one
asset also moves the expected returns of assets correlated with it.

### Limits

- No views ($k = 0$) or no confidence ($\Omega \to \infty$): $\mu_{BL} = \Pi$, the optimal portfolio is the market.
- Full confidence ($\Omega \to 0$): the views hold exactly, $P\mu_{BL} = q$.

### Portfolio weights

With $w_{BL} = (\delta\Sigma)^{-1}\mu_{BL}$ (ignoring parameter uncertainty), substituting the second form:

$$
w_{BL} = w_M + P^\top\Lambda, \qquad \Lambda = \frac1\delta\big(P\tau\Sigma P^\top + \Omega\big)^{-1}(q - P\Pi)\cdot\tau
$$

The portfolio is the market plus a combination of the **view portfolios** (rows of $P$), nothing else (He and Litterman 1999). Assets not in
any view keep their market weights.

### Choosing $\Omega$

A common choice (He–Litterman) is $\Omega = \operatorname{diag}(P\tau\Sigma P^\top)$, which gives each view the same weight as the prior. Idzorek (2005) maps a stated
confidence between 0% and 100% to $\Omega$ by requiring the tilt to be that fraction of the full-confidence tilt (exercise 5).

### Historical data as a view

Setting $P = I$, $q = \hat\mu$ (historical mean excess returns over $T$ years) and $\Omega = \hat\Sigma/T$ treats the sample as one big view. Black–Litterman then
shrinks the sample means toward equilibrium, with weights given by the relative precision of the two.

## 3. Worked Examples

### Example 1: scalar update

Prior: equity excess return $6\%$ with standard deviation $2\%$. View: $9\%$ with standard deviation $3\%$.

$$
E[\mu \mid q] = \frac{6/4 + 9/9}{1/4 + 1/9} = \frac{2.5}{0.3611} = 6.92\%, \qquad \operatorname{sd} = 1/\sqrt{0.3611} = 1.66\%
$$

The view moves the estimate by about 30% of the gap, because it is less precise than the prior.

### Example 2: reverse optimization

Three assets (Swiss equities, global equities, Swiss bonds) with market weights $(0.2, 0.4, 0.4)$ and $\delta = 2.5$:

$$
\Sigma w_M = (0.0256\cdot 0.2 + 0.02176\cdot 0.4 + 0.0008\cdot 0.4,\ \ 0.02176\cdot 0.2 + 0.0289\cdot 0.4,\ \ 0.0008\cdot 0.2 + 0.0025\cdot 0.4) = (0.01414,\ 0.01591,\ 0.00116)
$$

$\Pi = 2.5\,\Sigma w_M = (3.54\%,\ 3.98\%,\ 0.29\%)$ excess returns.

### Example 3: the update equals precision weighting

For one absolute view on asset 1 ($P = (1, 0, 0)$), the update of asset 1 is $\frac{\tau\sigma_1^2}{\tau\sigma_1^2 + \omega}(q - \Pi_1)$, the scalar formula. Asset 2 moves by $\frac{\tau\sigma_{12}}{\tau\sigma_1^2 + \omega}(q - \Pi_1)$: its
regression coefficient on asset 1 times asset 1's update.

## 4. Finance Application

### Where it matters

- **Strategic and tactical allocation.** Asset managers use Black–Litterman to turn house views into allocations that remain
  close to a neutral benchmark and change only where there is a view.
- **Communicating risk budgets.** The view portfolios $P$ and the multipliers $\Lambda$ show exactly which bets the portfolio takes.
- **Combining sources.** Equilibrium, analyst forecasts, quantitative signals and historical data can all enter as views with
  their own uncertainty.

### The usual misreadings

- "Black–Litterman predicts returns." It produces a consistent set of expected returns from a prior and views; the views still have to
  be good.
- "Without views the model returns the historical means." It returns $\Pi$, the equilibrium; historical means only enter if you put
  them in as views.
- "$\tau$ is just a technicality." Together with $\Omega$ it sets how far the views can move the prior; with He–Litterman's $\Omega$ only the ratio
  matters, but with an $\Omega$ from forecast errors, $\tau$ changes the result.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

assets = ["Swiss eq", "global eq", "EM eq", "Swiss small caps", "CHF bonds", "global bonds (hedged)", "CHF corporates", "Swiss real estate"]
N = len(assets)
vols = np.array([0.16, 0.17, 0.22, 0.19, 0.05, 0.045, 0.06, 0.12])
mu = np.array([0.065, 0.07, 0.08, 0.075, 0.02, 0.022, 0.028, 0.05])      # true expected returns (unknown in practice)
rf = 0.005
corr = np.eye(N)
for (i, j), v in {(0, 1): 0.80, (0, 2): 0.65, (0, 3): 0.85, (1, 2): 0.75, (1, 3): 0.70, (2, 3): 0.60,
                  (4, 5): 0.75, (4, 6): 0.85, (5, 6): 0.70}.items():
    corr[i, j] = corr[j, i] = v
for i in range(4):
    for j in range(4, 7):
        corr[i, j] = corr[j, i] = 0.25 if j == 6 else 0.10
    corr[i, 7] = corr[7, i] = 0.5
for j in range(4, 7):
    corr[j, 7] = corr[7, j] = 0.3
Sigma = np.outer(vols, vols) * corr
m_true = mu - rf

w_mkt = np.array([0.15, 0.30, 0.07, 0.03, 0.15, 0.20, 0.05, 0.05])      # market (benchmark) weights

### Equilibrium returns

Set $\delta$ so that the market portfolio's expected excess return is 4.0%, then reverse-optimize.

In [ ]:
mkt_excess = 0.04
delta = mkt_excess / (w_mkt @ Sigma @ w_mkt)
Pi = delta * Sigma @ w_mkt
print(f"market volatility {np.sqrt(w_mkt @ Sigma @ w_mkt):.2%}, delta {delta:.2f}")
print("optimal weights under Pi equal market weights:", np.allclose(np.linalg.solve(delta * Sigma, Pi), w_mkt))

R_hist = mu / 12 + rng.standard_normal((120, N)) @ np.linalg.cholesky(Sigma / 12).T
m_hist = R_hist.mean(axis=0) * 12 - rf
pd.DataFrame({"equilibrium Pi": Pi, "true (unknown)": m_true, "10-year historical": m_hist}, index=assets).style.format("{:.2%}")

With a market volatility of 9.6%, $\delta = 4.31$. The equilibrium returns are close to the true ones for the equity classes (within 0.8
points) and too low for bonds and real estate by 0.9–1.4 points: the market weights in this simulation are not exactly the optimal
portfolio for the true returns. The 10-year history is far worse: it shows negative excess returns for three of the four equity
classes, whose true premia are 6–7%.

### Two views

1. Absolute: EM equities will earn 9% in excess of cash (equilibrium says less).
2. Relative: Swiss small caps will beat Swiss large caps by 1.5% a year.

Uncertainty with the He–Litterman choice $\Omega = \operatorname{diag}(P\tau\Sigma P^\top)$, $\tau = 0.05$.

In [ ]:
def black_litterman(Pi, Sigma, P, q, Omega, tau):
    tS = tau * Sigma
    A = P @ tS @ P.T + Omega
    mu_bl = Pi + tS @ P.T @ np.linalg.solve(A, q - P @ Pi)
    M = tS - tS @ P.T @ np.linalg.solve(A, P @ tS)                   # posterior covariance of mu
    return mu_bl, M

def black_litterman_precision(Pi, Sigma, P, q, Omega, tau):
    tS_inv = np.linalg.inv(tau * Sigma)
    O_inv = np.linalg.inv(Omega)
    M = np.linalg.inv(tS_inv + P.T @ O_inv @ P)
    return M @ (tS_inv @ Pi + P.T @ O_inv @ q), M

tau = 0.05
P = np.zeros((2, N))
P[0, 2] = 1.0                                   # EM equities
P[1, 3], P[1, 0] = 1.0, -1.0                    # Swiss small caps minus Swiss eq
q = np.array([0.09, 0.015])
Omega = np.diag(np.diag(P @ (tau * Sigma) @ P.T))

mu_bl, M = black_litterman(Pi, Sigma, P, q, Omega, tau)
mu_bl2, _ = black_litterman_precision(Pi, Sigma, P, q, Omega, tau)
print("both formulas agree:", np.allclose(mu_bl, mu_bl2))
print(f"views implied by equilibrium: {P @ Pi}, stated: {q}, posterior: {(P @ mu_bl).round(4)}")

w_bl = np.linalg.solve(delta * Sigma, mu_bl)
bl_table = pd.DataFrame({"Pi": Pi, "BL posterior": mu_bl, "change": mu_bl - Pi,
                         "market weight": w_mkt, "BL weight": w_bl, "weight change": w_bl - w_mkt}, index=assets)
bl_table.style.format("{:.2%}")

In [ ]:
Lam = tau / delta * np.linalg.solve(P @ (tau * Sigma) @ P.T + Omega, q - P @ Pi)
print("w_BL - w_mkt = P' Lambda:", np.allclose(w_bl - w_mkt, P.T @ Lam), " Lambda =", Lam.round(4))

Equilibrium already implies 7.4% for EM equities and 0.4% for small caps over large caps. The posterior lands between equilibrium and
the views (8.3% and 1.0%), half way in each case, as the He–Litterman $\Omega$ implies. The weights confirm the theory: EM equities rise by 3.5
points, Swiss small caps rise by 12.4 points financed by Swiss large caps, and every other asset keeps its market weight, although
the expected returns of correlated assets (global equities, real estate) rise too.

### From no confidence to full confidence

Scale $\Omega$ from very large to very small and follow the EM weight and the small-cap tilt.

In [ ]:
scales = np.logspace(-3, 3, 61)
path = []
for c in scales:
    mu_c, _ = black_litterman(Pi, Sigma, P, q, Omega * c, tau)
    w_c = np.linalg.solve(delta * Sigma, mu_c)
    path.append({"scale": c, "EM eq weight": w_c[2], "Swiss small caps weight": w_c[3], "Swiss eq weight": w_c[0],
                 "EM excess return": mu_c[2]})
path = pd.DataFrame(path)
print(path.iloc[[0, 30, 60]].round(4).to_string(index=False))

With almost no confidence the portfolio is the market; with full confidence EM equities reach 13.5%, Swiss small caps 27%, and Swiss
large caps go short (−9%).

### Historical data as a view: Black–Litterman vs the plug-in

For 1,000 ten-year histories: treat the sample means as views with $\Omega = \hat\Sigma/T$, combine with the equilibrium prior for several $\tau$, and
compare the true Sharpe ratio of the resulting portfolios with the plug-in tangency, the market and 1/N.

In [ ]:
def true_sharpe(w):
    return (w @ m_true) / np.sqrt(w @ Sigma @ w)

L = np.linalg.cholesky(Sigma / 12)
T = 10
taus = [0.02, 0.1, 1 / T, 0.5, 2.0]
taus = sorted(set(taus))
rows = []
for h in range(1_000):
    R = mu / 12 + rng.standard_normal((T * 12, N)) @ L.T
    mh = R.mean(axis=0) * 12 - rf
    Sh = np.cov(R, rowvar=False) * 12
    Pi_h = delta * Sh @ w_mkt                      # equilibrium from the estimated covariance
    row = {"plug-in": true_sharpe(np.linalg.solve(Sh, mh))}
    for t_ in taus:
        mb, _ = black_litterman(Pi_h, Sh, np.eye(N), mh, Sh / T, t_)
        row[f"BL tau={t_:g}"] = true_sharpe(np.linalg.solve(Sh, mb))
    rows.append(row)
oos = pd.DataFrame(rows)
summary = pd.DataFrame({"mean true SR": oos.mean(), "5% quantile": oos.quantile(0.05)})
summary.loc["market (no data)"] = [true_sharpe(w_mkt), np.nan]
summary.loc["1/N"] = [true_sharpe(np.full(N, 1 / N)), np.nan]
summary.loc["true maximum"] = [np.sqrt(m_true @ np.linalg.solve(Sigma, m_true)), np.nan]
summary.round(3)

Combining the historical means with the equilibrium prior raises the average true Sharpe ratio from 0.28 (plug-in) to 0.47 with a
confident prior ($\tau = 0.02$), and the bad cases improve even more (5% quantile 0.40 against 0.01). But the best Black–Litterman version only
matches the market portfolio (0.466): ten years of returns add no usable information about expected returns here, and the more
weight they get (larger $\tau$), the worse the result. Black–Litterman's value lies in protecting the portfolio from noisy inputs and
in adding views that carry real information, not in extracting signal from short histories.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(N)
ax.bar(x_pos - 0.27, Pi, width=0.27, label="equilibrium Pi")
ax.bar(x_pos, m_true, width=0.27, label="true (unknown)")
ax.bar(x_pos + 0.27, m_hist, width=0.27, label="10-year historical")
ax.set_xticks(x_pos)
ax.set_xticklabels(assets, rotation=25, ha="right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Expected excess returns: equilibrium, truth and one 10-year history")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x_pos - 0.2, w_mkt, width=0.4, label="market weights")
ax.bar(x_pos + 0.2, w_bl, width=0.4, label="Black-Litterman weights (two views)")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(assets, rotation=25, ha="right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Only the assets in the views move")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for col in ["EM eq weight", "Swiss small caps weight", "Swiss eq weight"]:
    ax.semilogx(path["scale"], path[col], label=col)
ax.invert_xaxis()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Weights as confidence rises (Omega scaled from 1000× to 0.001× He-Litterman)")
ax.set_xlabel("Omega scale (log, decreasing = more confident)")
ax.set_ylabel("weight")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cols = list(oos.columns)
ax.boxplot([oos[c] for c in cols], showfliers=False)
ax.set_xticks(range(1, len(cols) + 1))
ax.set_xticklabels(cols, rotation=15)
ax.axhline(summary.loc["true maximum", "mean true SR"], color="black", linewidth=1, label="true maximum")
ax.axhline(summary.loc["market (no data)", "mean true SR"], color="tab:green", linestyle="--", label="market portfolio")
ax.set_title("True Sharpe ratio: historical means alone vs combined with the equilibrium prior")
ax.set_ylabel("annual Sharpe ratio")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. Prior 3% with standard deviation 1%, view 1% with standard deviation 1%. Compute the posterior mean and standard deviation.
2. Two assets with volatilities 16% and 5%, correlation 0.1, market weights 50/50, $\delta = 3$. Compute $\Pi$.
3. After adding a view that EM equities will outperform, the Black–Litterman portfolio has 8 more points in EM equities and every other
   weight unchanged, although the expected returns of several other assets also rose. **Interpret** why only this weight moves, and what
   the investment committee is actually deciding when it sets the view's confidence.

### Intermediate

4. Show numerically that the two posterior formulas give the same $\mu_{BL}$ and $M$ for random views ($k = 3$, random $P$, $q$, diagonal $\Omega$).
5. Idzorek's method: find the scale of $\Omega_{11}$ (EM view alone) at which the EM weight tilt is 50% of the full-confidence tilt.

### Advanced

6. Show that with $P = I$, $q = \hat\mu$, $\Omega = \Sigma/T$ and $\tau = 1/T$, the posterior is $\mu_{BL} = (\Pi + \hat\mu)/2$. More generally, show $\mu_{BL} = \frac{T}{T + 1/\tau}\hat\mu + \frac{1/\tau}{T + 1/\tau}\Pi$.
7. In the historical-data experiment, why does a small $\tau$ come close to the market portfolio, and why does the market portfolio
   itself fall short of the true maximum here? What would have to be true for the market to be optimal?

## 8. Solutions / Checks

In [ ]:
# 1.
post = (3 / 1 + 1 / 1) / (1 / 1 + 1 / 1)
print(f"1. posterior mean {post:.1f}%, sd {1 / np.sqrt(2):.2f}%")

# 2.
S2 = np.array([[0.0256, 0.1 * 0.16 * 0.05], [0.1 * 0.16 * 0.05, 0.0025]])
print(f"2. Pi = {(3 * S2 @ np.array([0.5, 0.5])).round(5)}")

# 4.
P4 = rng.standard_normal((3, N))
q4 = rng.normal(0.03, 0.02, 3)
O4 = np.diag(rng.uniform(0.0005, 0.003, 3))
a1, M1 = black_litterman(Pi, Sigma, P4, q4, O4, tau)
a2, M2 = black_litterman_precision(Pi, Sigma, P4, q4, O4, tau)
print(f"4. means agree {np.allclose(a1, a2)}, covariances agree {np.allclose(M1, M2)}")

In [ ]:
# 5.
P5, q5 = P[:1], q[:1]
O5 = np.diag(np.diag(P5 @ (tau * Sigma) @ P5.T))
w_full = np.linalg.solve(delta * Sigma, black_litterman(Pi, Sigma, P5, q5, O5 * 1e-9, tau)[0])
full_tilt = w_full[2] - w_mkt[2]
from scipy.optimize import brentq
def tilt_gap(log_c):
    w_c = np.linalg.solve(delta * Sigma, black_litterman(Pi, Sigma, P5, q5, O5 * np.exp(log_c), tau)[0])
    return (w_c[2] - w_mkt[2]) - 0.5 * full_tilt
c50 = np.exp(brentq(tilt_gap, -10, 10))
print(f"5. full-confidence EM tilt {full_tilt:.2%}; 50% of it at Omega scale {c50:.3f} (He-Litterman = 1)")

# 6.
mh6 = m_hist
mb6, _ = black_litterman(Pi, Sigma, np.eye(N), mh6, Sigma / T, 1 / T)
print(f"6. mu_BL == (Pi + mu_hat)/2: {np.allclose(mb6, (Pi + mh6) / 2)}")
for t_ in [0.02, 0.5]:
    mb, _ = black_litterman(Pi, Sigma, np.eye(N), mh6, Sigma / T, t_)
    wgt = T / (T + 1 / t_)
    print(f"   tau {t_}: weight on data {wgt:.3f}, formula holds {np.allclose(mb, wgt * mh6 + (1 - wgt) * Pi)}")

**1.** Equal precisions, so the posterior mean is the average, 2%, with standard deviation $1/\sqrt2 = 0.71\%$.

**2.** $\Sigma w = (0.0256\cdot 0.5 + 0.0008\cdot 0.5,\ 0.0008\cdot 0.5 + 0.0025\cdot 0.5) = (0.0132, 0.00165)$; $\Pi = 3\cdot\Sigma w = (3.96\%, 0.50\%)$.

**3.** Black–Litterman weights are the market weights plus multiples of the view portfolios, and an absolute view on EM equities has
the view portfolio "EM only". The other expected returns rise because they are correlated with EM, but exactly by the amount that
leaves their optimal weights unchanged given their higher covariance with the larger EM position. Setting the confidence decides the
**size** of the active bet: with low confidence the portfolio stays near the benchmark, with full confidence it takes the full bet
implied by the view at the given risk aversion.

**4.** Both formulas agree to machine precision.

**5.** The 50% tilt is reached when $\Omega$ for the view equals its He–Litterman value (scale ≈ 1): with $\Omega = P\tau\Sigma P^\top$ the view and the prior
have the same precision for that portfolio, so the posterior goes half way.

**6.** With $P = I$: $\mu_{BL} = [(\tau\Sigma)^{-1} + T\Sigma^{-1}]^{-1}[(\tau\Sigma)^{-1}\Pi + T\Sigma^{-1}\hat\mu] = \frac{1}{1/\tau + T}\big[\Pi/\tau + T\hat\mu\big]$, because every term is a multiple of $\Sigma^{-1}$.
With $\tau = 1/T$ the two weights are equal.

**7.** A small $\tau$ means a confident prior: the data get weight $T/(T + 1/\tau)$, small for $\tau = 0.02$, so the portfolio is close to the market.
The market falls short of the true maximum because in this simulation the true expected returns are **not** the equilibrium
returns implied by $w_M$: the market portfolio would be optimal only if $\mu - r_f = \delta\Sigma w_M$, i.e. if everyone's expectations were correct
and identical (CAPM). Black–Litterman helps exactly to the extent that the data or the views point toward the truth.

## 9. Conclusion

- Equilibrium returns $\Pi = \delta\Sigma w_M$ are the expected returns under which the market is optimal; they are the prior.
- Views $P\mu = q + \varepsilon$ with uncertainty $\Omega$ are the data; the posterior mean is $\Pi + \tau\Sigma P^\top(P\tau\Sigma P^\top + \Omega)^{-1}(q - P\Pi)$, a precision-weighted
  combination.
- The optimal portfolio is the market plus the view portfolios; confidence sets the size of each bet.
- Historical means entered as a view are shrunk toward equilibrium; this removes most of the damage of the plug-in portfolio
  (08/01), but ten years of data do not improve on the market portfolio itself. Good views, not short histories, are what can add value.

This closes module 08. Next: [Module 09, 01 Random walk and autocorrelation](../09_time_series_and_volatility/01_random_walk_and_autocorrelation.ipynb).